In [ ]:
# ============================================================
# BIASCOPE — Notebook 11 (v2): Uncertainty-aware prediction + Cross-lingual consistency
# FIX: val rows me row_id duplicate hote hain (MBIB en+hi) => val ko position se align karo
# CPU only. Uses existing predictions (no retraining).
# ============================================================
!pip -q install -U sentence-transformers

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, pearsonr, rankdata
from sklearn.metrics import roc_auc_score
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
SRC = BASE_DIR / "outputs" / "nb07b_babe"
BABE_DIR = BASE_DIR / "data" / "multilingual" / "babe"
OUT = BASE_DIR / "outputs" / "nb11_uncertainty_consistency"
FINAL_DIR = BASE_DIR / "models" / "final"
OUT.mkdir(parents=True, exist_ok=True)

SEEDS = (42, 43, 44)
RUN_FMT = "xlmr|lora|mbib+babe|s{}"
EVALS = ["mbib_en_test", "mbib_hi_test", "babe_test_en", "babe_test_hi", "gold_conf2"]
ABSTAIN_MARGIN_Q = 0.15     # val pe sabse kam margin wale 15% abstain
ABSTAIN_STD_Q = 0.90        # val pe seed-disagreement ke top 10% abstain
N_BOOT = 500
EPS = 1e-6
rng = np.random.default_rng(0)

P = pd.read_csv(SRC / "predictions_07c.csv")
runs = {s: RUN_FMT.format(s) for s in SEEDS}
missing = [r for r in runs.values() if r not in set(P["run_id"])]
assert not missing, f"Predictions missing for: {missing}"

# ---------------- helpers ----------------
def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def nll(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
def fit_T(y, p):
    z = logit(p)
    r = minimize_scalar(lambda lt: nll(y, sigmoid(z / np.exp(lt))), bounds=(-3, 3), method="bounded")
    return float(np.exp(r.x))

def get_raw(eval_set):
    cols, y, lens = {}, None, set()
    for s, r in runs.items():
        d = P[(P["run_id"] == r) & (P["eval_set"] == eval_set)].reset_index(drop=True)
        if eval_set != "val":                 # test sets me row_id unique hain
            d = d.drop_duplicates("row_id")
            d.index = d["row_id"]
        lens.add(len(d))                      # val me position se align (same order har run me)
        cols[s] = d["p"]
        if y is None:
            y = d["y"]
    assert len(lens) == 1, f"{eval_set}: seeds ke beech rows ka count alag hai: {lens}"
    M = pd.DataFrame(cols).dropna()
    return M, y.loc[M.index].astype(int)

# temperatures per seed (poore validation pe fit)
Mv_raw, yv = get_raw("val")
print("val rows per seed:", len(Mv_raw))
T = {s: fit_T(yv.values, Mv_raw[s].values) for s in SEEDS}
print("Per-seed temperatures:", {s: round(t, 4) for s, t in T.items()})
print("(s42 ki T ab 08a ke 'Final temperature saved' wali value ke barabar honi chahiye)")

def build(eval_set):
    M_raw, y = get_raw(eval_set)
    M = M_raw.copy()
    for s in SEEDS:
        M[s] = sigmoid(logit(M_raw[s].values) / T[s])
    return M, y

def uncertainty(M):
    p = M.mean(axis=1).values
    return p, M.std(axis=1, ddof=0).values, 1 - 2 * np.abs(p - 0.5)   # p_ens, seed_std, margin_uncertainty

# ---------------- thresholds from VALIDATION ----------------
Mv, yv = build("val")
pv, sv, _ = uncertainty(Mv)
margin_delta = float(np.quantile(np.abs(pv - 0.5), ABSTAIN_MARGIN_Q))
std_sigma = float(np.quantile(sv, ABSTAIN_STD_Q))
print(f"Thresholds (from val): abstain if |p-0.5| < {margin_delta:.3f}  OR  seed_std > {std_sigma:.3f}")

# ============================================================
# A. UNCERTAINTY-AWARE PREDICTION (selective classification)
# ============================================================
def sel_acc(y, p, u, cov):
    k = max(1, int(round(cov * len(y))))
    idx = np.argsort(u, kind="stable")[:k]
    return float(((p[idx] >= 0.5).astype(int) == y[idx]).mean())

def aurc(y, p, u):
    idx = np.argsort(u, kind="stable")
    wrong = ((p[idx] >= 0.5).astype(int) != y[idx]).astype(float)
    return float(np.mean(np.cumsum(wrong) / np.arange(1, len(y) + 1)))

rows, curves = [], {}
for ev in EVALS:
    M, y = build(ev)
    p, sd, um = uncertainty(M)
    yy = y.values
    u_comb = (rankdata(um) + rankdata(sd)) / (2 * len(yy))
    measures = {"margin": um, "seed_std": sd, "combined": u_comb}
    acc_all = float(((p >= 0.5).astype(int) == yy).mean())
    curves[ev] = (yy, p, measures, acc_all)
    for name, u in measures.items():
        row = dict(eval_set=ev, n=len(yy), measure=name, acc_all=acc_all, aurc=aurc(yy, p, u),
                   aurc_random=1 - acc_all)
        for c in (0.9, 0.8, 0.6):
            row[f"acc@{c}"] = sel_acc(yy, p, u, c)
        gains = []
        for _ in range(N_BOOT):
            i = rng.integers(0, len(yy), len(yy))
            gains.append(sel_acc(yy[i], p[i], u[i], 0.8) - float(((p[i] >= 0.5).astype(int) == yy[i]).mean()))
        row["gain@0.8"] = row["acc@0.8"] - acc_all
        row["gain_ci"] = f"[{np.percentile(gains, 2.5):.3f}, {np.percentile(gains, 97.5):.3f}]"
        rows.append(row)
SEL = pd.DataFrame(rows)
print("\n" + "=" * 70 + "\nA1. SELECTIVE ACCURACY (abstain on most uncertain samples)\n" + "=" * 70)
display(SEL.round(3))
SEL.to_csv(OUT / "selective_classification.csv", index=False)

# operational policy: abstain if margin < delta OR seed_std > sigma (thresholds from val)
pol = []
for ev in EVALS:
    M, y = build(ev)
    p, sd, _ = uncertainty(M)
    yy = y.values
    unc = (np.abs(p - 0.5) < margin_delta) | (sd > std_sigma)
    keep = ~unc
    corr = ((p >= 0.5).astype(int) == yy)
    pol.append(dict(eval_set=ev, n=len(yy), abstain_rate=float(unc.mean()), acc_all=float(corr.mean()),
                    acc_retained=float(corr[keep].mean()), acc_abstained=float(corr[unc].mean()) if unc.any() else np.nan,
                    auc_retained=roc_auc_score(yy[keep], p[keep]) if len(set(yy[keep])) == 2 else np.nan))
POL = pd.DataFrame(pol).set_index("eval_set")
print("\n" + "=" * 70 + "\nA2. ABSTAIN POLICY (val-derived thresholds)\n" + "=" * 70)
display(POL.round(3))
print("abstain_rate gold/Hindi me in-domain se zyada ho to model shift pehchanta hai; warna sirf sample-level difficulty flag hoti hai.")
POL.to_csv(OUT / "abstain_policy.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ev in zip(axes, ["babe_test_en", "babe_test_hi", "gold_conf2"]):
    yy, p, meas, acc_all = curves[ev]
    cov = np.linspace(0.3, 1.0, 36)
    for name, u in meas.items():
        ax.plot(cov, [sel_acc(yy, p, u, c) for c in cov], label=name)
    ax.axhline(acc_all, color="gray", ls="--", label="no abstention")
    ax.set_title(ev); ax.set_xlabel("coverage"); ax.set_ylabel("accuracy on kept samples"); ax.legend()
plt.tight_layout()
plt.savefig(OUT / "risk_coverage.png", dpi=150)
plt.show()

json.dump(dict(adapters=[f"xlmr_lora_mbib_babe_s{s}" for s in SEEDS], temperatures={str(s): T[s] for s in SEEDS},
               margin_delta=margin_delta, seed_std_sigma=std_sigma,
               abstain_quantiles=dict(margin=ABSTAIN_MARGIN_Q, seed_std=ABSTAIN_STD_Q),
               note="thresholds fit on full mbib+babe validation set; policy: uncertain if |p_ens-0.5|<margin_delta OR seed_std>seed_std_sigma"),
          open(FINAL_DIR / "uncertainty_config.json", "w"), indent=2)
print("\nSaved demo config:", FINAL_DIR / "uncertainty_config.json")

# ============================================================
# B. CROSS-LINGUAL CONSISTENCY (BABE test: English vs NLLB Hindi)
# ============================================================
Me, ye = build("babe_test_en")
Mh, yh = build("babe_test_hi")
pe, ph = Me.mean(axis=1), Mh.mean(axis=1)
pe.index = pe.index.str.replace(r"_en$", "", regex=True)
ph.index = ph.index.str.replace(r"_hi$", "", regex=True)
ye.index = ye.index.str.replace(r"_en$", "", regex=True)
common = pe.index.intersection(ph.index)
pe, ph, y = pe.loc[common], ph.loc[common], ye.loc[common].values
d = ph.values - pe.values
agree = float(((pe.values >= 0.5) == (ph.values >= 0.5)).mean())
y1 = (y == 1) & (pe.values >= 0.5)
res = dict(n_pairs=len(common), label_agreement=agree,
           spearman=float(spearmanr(pe, ph).correlation), pearson=float(pearsonr(pe, ph)[0]),
           mean_delta_hi_minus_en=float(d.mean()), mean_abs_delta=float(np.abs(d).mean()),
           share_abs_delta_gt_0p25=float((np.abs(d) > 0.25).mean()),
           auc_en=roc_auc_score(y, pe.values), auc_hi=roc_auc_score(y, ph.values),
           flip_rate_when_en_flags_biased=float((ph.values[y1] < 0.5).mean()))
print("\n" + "=" * 70 + "\nB. CROSS-LINGUAL CONSISTENCY (same sentence, English vs Hindi)\n" + "=" * 70)
display(pd.Series(res).round(3).to_frame("value"))

# translation quality (LaBSE) vs shift
from sentence_transformers import SentenceTransformer
bh = pd.read_csv(BABE_DIR / "babe_hi.csv", encoding="utf-8-sig")
bh["uuid"] = bh["uuid"].astype(str)
bh = bh[bh["split"] == "test"].drop_duplicates("uuid").set_index("uuid").loc[common]
labse = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
a = labse.encode(bh["text"].astype(str).tolist(), normalize_embeddings=True, batch_size=32, show_progress_bar=False)
b = labse.encode(bh["text_hi"].astype(str).tolist(), normalize_embeddings=True, batch_size=32, show_progress_bar=False)
sim = (a * b).sum(1)
rho = spearmanr(np.abs(d), sim).correlation
q = pd.qcut(sim, 4, labels=["Q1 (lowest sim)", "Q2", "Q3", "Q4 (highest sim)"])
tab = pd.DataFrame({"abs_delta": np.abs(d), "sim": sim, "q": q}).groupby("q", observed=True).agg(
    n=("abs_delta", "size"), mean_abs_delta=("abs_delta", "mean"), mean_sim=("sim", "mean"))
print(f"\nSpearman(|Δp|, LaBSE similarity) = {rho:.3f}  (negative => kharab translation zyada shift deta hai)")
display(tab.round(3))

ex = pd.DataFrame({"en": bh["text"].values, "hi": bh["text_hi"].values, "p_en": pe.values, "p_hi": ph.values,
                   "delta": d, "labse_sim": sim, "label": y}, index=common)
ex = ex.reindex(ex["delta"].abs().sort_values(ascending=False).index).head(10)
pd.set_option("display.max_colwidth", 110)
print("\nLargest English→Hindi shifts (qualitative):")
display(ex.round(2))
ex.to_csv(OUT / "largest_translation_shifts.csv", encoding="utf-8-sig")
pd.Series(res).to_csv(OUT / "cross_lingual_consistency.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(pe, ph, s=8, alpha=0.4); ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set_xlabel("P(biased) English"); ax[0].set_ylabel("P(biased) Hindi (NLLB translation)")
ax[0].set_title(f"Same sentence, two languages (Spearman {res['spearman']:.2f})")
ax[1].hist(d, bins=40); ax[1].set_xlabel("Δ = P_hi − P_en"); ax[1].set_title("Translation-induced shift")
plt.tight_layout()
plt.savefig(OUT / "cross_lingual_consistency.png", dpi=150)
plt.show()

print("\nSaved in:", OUT)
print("""
KAISE PADHNA / REPORT ME:
- A1: gain@0.8 > 0 aur CI zero se door => uncertainty ranking sahi hai (abstain karne se accuracy badhti hai).
- A2: acc_abstained < acc_retained hona chahiye. abstain_rate gold pe in-domain se zyada nahi hai to
  'domain shift detect karta hai' mat likhna; sirf 'sample-level difficulty' likho.
- Thresholds poore val se fit hue (test leakage nahi). Single model-family ka 3-seed ensemble hai.
- B: label_agreement/spearman => classifier translation ke baad bhi bias signal kitna preserve karta hai.
  Hindi labels MT se aaye hain; 'shift' me classifier ka bias + translation ka meaning-change dono mix hain.
""")